# SAA+ Benchmark on MVTec and VisA

Reproduces image-AUROC / pixel-AUROC / AP / F1 from the SAA+ paper  
(~500 images per dataset, stratified sampling, T4 GPU)

In [ ]:
%cd /content
!git clone -b dev https://github.com/SyDuc7421/Segment-Any-Anomaly.git
%cd Segment-Any-Anomaly/

import re, pathlib

# Patch 1: remove stale transformers upper-bound in GroundingDINO setup
for p in pathlib.Path('GroundingDINO').rglob('*.py'):
    txt = p.read_text()
    patched = re.sub(r'transformers[^"\']*<4\.\d+\.\d+', 'transformers>=4.41.0', txt)
    if patched != txt:
        p.write_text(patched)

# Patch 2: fix BertModel.get_head_mask removed in transformers 5.x
bw = pathlib.Path('GroundingDINO/groundingdino/models/GroundingDINO/bertwarper.py')
txt = bw.read_text()
old = 'self.get_head_mask = bert_model.get_head_mask'
new = (
    'if hasattr(bert_model, "get_head_mask"):\n'
    '            self.get_head_mask = bert_model.get_head_mask\n'
    '        else:\n'
    '            def _get_head_mask(head_mask, num_hidden_layers, is_attention_chunked=False):\n'
    '                if head_mask is not None:\n'
    '                    if head_mask.dim() == 1:\n'
    '                        head_mask = head_mask.unsqueeze(0).unsqueeze(0).unsqueeze(-1).unsqueeze(-1)\n'
    '                        head_mask = head_mask.expand(num_hidden_layers, -1, -1, -1, -1)\n'
    '                    elif head_mask.dim() == 2:\n'
    '                        head_mask = head_mask.unsqueeze(1).unsqueeze(-1).unsqueeze(-1)\n'
    '                    if is_attention_chunked:\n'
    '                        head_mask = head_mask.unsqueeze(-1)\n'
    '                else:\n'
    '                    head_mask = [None] * num_hidden_layers\n'
    '                return head_mask\n'
    '            self.get_head_mask = _get_head_mask'
)
if old in txt:
    bw.write_text(txt.replace(old, new))

%cd GroundingDINO/
!pip install -q -e . --no-build-isolation
%cd ../SAM
!pip install -q -e .
!pip install -q "transformers>=4.41.0" "supervision>=0.6.0,<0.21.0" \
    opencv-python pycocotools matplotlib onnxruntime onnx ipykernel gradio loguru
%cd ..

In [ ]:
# Restart so updated transformers is loaded from disk
import os
os.kill(os.getpid(), 9)

In [ ]:
%cd /content/Segment-Any-Anomaly
%mkdir -p weights
%cd weights
!wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
!wget -q https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth
%cd ..

In [ ]:
%cd /content/Segment-Any-Anomaly
%mkdir -p /content/datasets

from google.colab import userdata
import json, pathlib, os

pathlib.Path('/root/.kaggle').mkdir(exist_ok=True)
pathlib.Path('/root/.kaggle/kaggle.json').write_text(json.dumps({
    'username': userdata.get('KAGGLE_USERNAME'),
    'key': userdata.get('KAGGLE_KEY')
}))
!chmod 600 /root/.kaggle/kaggle.json

# Accept dataset terms first at: https://www.kaggle.com/datasets/ipythonx/mvtec-ad
!pip install -q kaggle
!kaggle datasets download -d ipythonx/mvtec-ad -p /content/datasets/ --unzip

os.environ['MVTEC_DIR'] = '/content/datasets'

from datasets import mvtec_classes
present = [c for c in mvtec_classes if os.path.isdir(f'/content/datasets/{c}')]
print(f'MVTec: {len(present)}/15 classes ready:', present)

In [ ]:
%cd /content/Segment-Any-Anomaly
%mkdir -p /content/datasets

!wget -q --show-progress \
    "https://amazon-visual-anomaly.s3.us-west-2.amazonaws.com/VisA_20220922.tar" \
    -O /content/datasets/visa.tar
!tar -xf /content/datasets/visa.tar -C /content/datasets/

!python datasets/prepare_visa_public.py \
    --data-folder /content/datasets \
    --save-folder /content/datasets/VisA_pytorch \
    --split-file /content/datasets/split_csv/1cls.csv

import os
os.environ['VISA_DIR'] = '/content/datasets/VisA_pytorch/1cls'
print('VisA classes:', sorted(os.listdir(os.environ['VISA_DIR'])))

In [ ]:
%cd /content/Segment-Any-Anomaly
import os, subprocess

os.environ['MVTEC_DIR'] = '/content/datasets'

result = subprocess.run(
    ['python', 'eval_SAA.py',
     '--dataset', 'mvtec',
     '--class-name', 'carpet',
     '--batch-size', '1',
     '--root-dir', './result',
     '--cal-pro', 'False',
     '--gpu-id', '0',
     '--max-samples', '2'],
    cwd='/content/Segment-Any-Anomaly',
    capture_output=True, text=True
)
print(result.stdout[-3000:] if result.stdout else '(no stdout)')
print('STDERR:', result.stderr[-3000:] if result.stderr else '(no stderr)')
print('exit code:', result.returncode)

In [ ]:
%cd /content/Segment-Any-Anomaly
import os, subprocess

os.environ['VISA_DIR'] = '/content/datasets/VisA_pytorch/1cls'
os.environ['MAX_SAMPLES'] = '42'  # ~504 images across 12 classes

result = subprocess.run(
    ['python', 'run_VisA_public.py'],
    cwd='/content/Segment-Any-Anomaly'
)
print('VisA benchmark done, exit code:', result.returncode)

In [ ]:
%cd /content/Segment-Any-Anomaly
import pandas as pd, glob

def summarize(csv_path, label):
    df = pd.read_csv(csv_path, index_col=0)
    mean_row = df.mean(numeric_only=True).rename('MEAN')
    df = pd.concat([df, mean_row.to_frame().T])
    cols = [c for c in ['i_roc','p_roc','i_ap','p_ap','i_f1','p_f1'] if c in df.columns]
    print(f'\n{"="*60}\n  {label}\n{"="*60}')
    print(df[cols].to_string(float_format='{:.2f}'.format))

for p in glob.glob('result/csv/mvtec-indx-*.csv'):
    summarize(p, f'MVTec — {p}')
for p in glob.glob('result/csv/visa_public-indx-*.csv'):
    summarize(p, f'VisA — {p}')

In [ ]:
from google.colab import drive
import shutil, glob, os

drive.mount('/content/drive')
dst = '/content/drive/MyDrive/SAA_results'
os.makedirs(dst, exist_ok=True)

for p in glob.glob('/content/Segment-Any-Anomaly/result/csv/*.csv'):
    shutil.copy(p, dst)
    print('Saved:', os.path.basename(p))

print('Done → Google Drive/SAA_results/')